In [30]:
# imports
import numpy as np
import h5py
from scipy.stats import sem
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

from functions import neuron_reliability, reliability_filtering_tb

Get the Tau values from the fit of the exponential decay function to neural data

In [2]:
# define the exponential decay function 
def adapt_resp_decay(x, C, T):
    return C*np.exp(-x/T)

In [5]:
# load the responses for S5 (no adaptation) - the aim is to use these to get reliable neurons for S5 during the tb from 70-170 ms
file_s5 = h5py.File('data/230801.pico.rsvp.gratingsAdap_s5.experiment_psth_raw.h5','r')

rates_s5 = file_s5['/psth'][:]

s5_neural = np.transpose(np.nanmean(rates_s5[:,:,7:17,:],axis=2), (0,2,1))

/tmp/ipykernel_2516559/2408203082.py:6: RuntimeWarning: Mean of empty slice
  s5_neural = np.transpose(np.nanmean(rates_s5[:,:,7:17,:],axis=2), (0,2,1))


In [7]:
# get the array of neuron reliabilities during S5
rel = neuron_reliability(s5_neural)[0]

In [9]:
#load and format S3 (right motion adaptation) & S4 (left motion adaptation)
file_s3 = h5py.File('data/230801.pico.rsvp.gratingsAdap_s3.experiment_psth_raw.h5','r')
file_s4 = h5py.File('data/230801.pico.rsvp.gratingsAdap_s4.experiment_psth_raw.h5','r')

rates_s3 = file_s3['/psth'][:]
rates_s4 = file_s4['/psth'][:]

In [12]:
# get only the reliable neurons for S3 and S4
rates_s3 = reliability_filtering_tb(rates_s3, rel, metric=0.2)
rates_s4 = reliability_filtering_tb(rates_s4, rel, metric=0.2)

In [15]:
# get the neural data for fitting the decay function by averaging across images and selecting only 100-1100 ms and averaging across trials
neural_s3 = np.nanmean(np.nanmean(rates_s3[:,:,10:110,:], axis=0), axis=0)
neural_s4 = np.nanmean(np.nanmean(rates_s4[:,:,10:110,:], axis=0), axis=0)

neural_s3_se = sem(np.nanmean(rates_s3[:,:,10:110,:],axis=1), axis=0, nan_policy='omit')
neural_s4_se = sem(np.nanmean(rates_s4[:,:,10:110,:],axis=1), axis=0, nan_policy='omit')

In [22]:
# set up parameters
n_neurons = neural_s3.shape[1]
n_seasons = 2

In [23]:
# make an array to hold the tau and constant for each neuron
tau_arr = np.empty((n_neurons,n_seasons,2)) # 0 - tau, 1 - contstant C

In [24]:
# fit the decay function to neural data for each neuron and trial
x = np.arange(0,1000,10) # the times that we are providing as inputs for our function
seasons = [neural_s3, neural_s4]

for s in range(n_seasons):
    season_data = seasons[s]
    for n in range(n_neurons):
        resp = season_data[:, n] # the neural responses for each time point
        c_val, tau_val = curve_fit(adapt_resp_decay,xdata=x,ydata=resp)[0] # the function gives us the fitted coefficient and tau the p term we discarded p0=[resp[0],1]
        tau_arr[n, s, 0] = tau_val
        tau_arr[n, s, 1] = c_val 

In [38]:
# save the tau arr
np.save('data/tau_arr.npy',tau_arr)

Evaluate the fit of the decay function to the neural data

In [25]:
# first lets write a function to generate an array of Tau model data of equal size to the neural data used for fitting the Tau function (100,72) for each season
x = np.arange(0,1000,10) # the times that we are providing as inputs for our function

def decay_func_data(season_tau, season_C, n_neuron=n_neurons, time=x):
    mod_data = np.empty((time.shape[0],n_neuron))
    for n in range(n_neuron):
        neuron_tau = season_tau[n]
        neuron_C = season_C[n]
        for t in range(time.shape[0]):
            mod_data[t,n] = adapt_resp_decay(time[t], neuron_C, neuron_tau)
    return mod_data

In [27]:
# now lets use our function
pred_s3 = decay_func_data(tau_arr[:,0,0], tau_arr[:,0,1])
pred_s4 = decay_func_data(tau_arr[:,1,0], tau_arr[:,1,1])

In [28]:
# define a function to generate an r2 score for each unit in a season
def decay_func_fit(neural_data, func_data):
    r2s = np.empty(n_neurons)
    for n in range(n_neurons):
        r2s[n] = r2_score(neural_data[:,n],func_data[:,n])
    return r2s

In [31]:
# get the r2_score that evaluates the fit of the exponential decay function to the unit's time series of responses
r2s_s3 = decay_func_fit(neural_s3, pred_s3)
r2s_s4 = decay_func_fit(neural_s4, pred_s4)

In [32]:
# create an index that only includes units with good r2 for either s3 or s4 (or both)
good_r2_idx = np.squeeze(np.argwhere((r2s_s3 > 0.3) | (r2s_s4 > 0.3)))

In [34]:
# filter tau array to include only Tau for S3 and S4 for units that are reliable
tau_arr_s34_good_r2 = tau_arr[good_r2_idx,:,0]

In [37]:
# save for later use
np.save('data/tau_arr_s34_good_r2', tau_arr_s34_good_r2)